# Giai đoạn 1 - Chuẩn bị và Làm sạch Dữ liệu

Notebook này được scaffold sẵn cho Giai đoạn 1.

Mục tiêu:
- Đọc 9 file CSV trong `raw_data/`
- Tạo các bảng trung gian đã làm sạch
- Tạo 2 analytical mart cuối cùng:
  - `mart_order_level`: 1 dòng = 1 order
  - `mart_item_level`: 1 dòng = 1 order item
- Lưu tất cả output vào `clean_data/`

Lưu ý quan trọng:

- Không merge trực tiếp `payments` và `reviews` vào `order_items` trước khi aggregate.


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

# Đặt tùy chọn hiển thị để dễ debug khi chạy thủ công.
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 140)


## 1. Path config

Cell này xác định đường dẫn dự án, thư mục raw data, và thư mục output.
Notebook được thiết kế để chạy từ root của project.


In [2]:
# Dùng current working directory làm root của project.
# Nếu bạn mở notebook từ nơi khác, có thể sửa lại PROJECT_ROOT cho phù hợp.
PROJECT_ROOT = Path.cwd()
RAW_DIR = PROJECT_ROOT / "raw_data"
OUTPUT_DIR = PROJECT_ROOT / "clean_data"

# Tạo thư mục output nếu chưa tồn tại.
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("RAW_DIR:", RAW_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)


PROJECT_ROOT: /Users/tranquangtrong/Desktop/Olist-InsightHub-From-Data-Exploration-to-Strategic-Storytelling
RAW_DIR: /Users/tranquangtrong/Desktop/Olist-InsightHub-From-Data-Exploration-to-Strategic-Storytelling/raw_data
OUTPUT_DIR: /Users/tranquangtrong/Desktop/Olist-InsightHub-From-Data-Exploration-to-Strategic-Storytelling/clean_data


## 2. Load raw data

Cell này đọc 9 file CSV gốc. Các cột timestamp được parse thành datetime ngay từ đầu.


In [3]:
# Đọc bảng orders và parse các cột timestamp quan trọng.
orders = pd.read_csv(
    RAW_DIR / "olist_orders_dataset.csv",
    parse_dates=[
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date",
    ],
)

# Đọc order items ở cấp item trong đơn hàng.
order_items = pd.read_csv(
    RAW_DIR / "olist_order_items_dataset.csv",
    parse_dates=["shipping_limit_date"],
)

# Đọc payments, reviews, customers, sellers, products, translation, geolocation.
payments = pd.read_csv(RAW_DIR / "olist_order_payments_dataset.csv")
reviews = pd.read_csv(
    RAW_DIR / "olist_order_reviews_dataset.csv",
    parse_dates=["review_creation_date", "review_answer_timestamp"],
)
customers = pd.read_csv(RAW_DIR / "olist_customers_dataset.csv")
sellers = pd.read_csv(RAW_DIR / "olist_sellers_dataset.csv")
products = pd.read_csv(RAW_DIR / "olist_products_dataset.csv")
translation = pd.read_csv(
    RAW_DIR / "product_category_name_translation.csv",
    encoding="utf-8-sig",
)
geolocation = pd.read_csv(RAW_DIR / "olist_geolocation_dataset.csv")

print("orders:", orders.shape)
print("order_items:", order_items.shape)
print("payments:", payments.shape)
print("reviews:", reviews.shape)
print("customers:", customers.shape)
print("sellers:", sellers.shape)
print("products:", products.shape)
print("translation:", translation.shape)
print("geolocation:", geolocation.shape)


orders: (99441, 8)
order_items: (112650, 7)
payments: (103886, 5)
reviews: (99224, 7)
customers: (99441, 5)
sellers: (3095, 4)
products: (32951, 9)
translation: (71, 2)
geolocation: (1000163, 5)


## 3. Helper functions và profile nhanh

Cell này tạo các helper để:
- lấy mode an toàn cho city/state/payment type
- tổng hợp thông tin cơ bản của mỗi bảng
- để bạn kiểm tra null, unique key, và tình trạng raw data trước khi merge


In [4]:
def safe_mode(series: pd.Series):
    """Trả về mode đầu tiên sau khi bỏ null và strip text."""
    cleaned = series.dropna().astype(str).str.strip()
    if cleaned.empty:
        return pd.NA
    modes = cleaned.mode()
    return modes.iloc[0] if not modes.empty else pd.NA


def summarize_table(name: str, df: pd.DataFrame, key_col: str | None = None) -> dict:
    """Tạo summary ngắn gọn để kiểm tra số dòng, số cột và unique key."""
    row = {
        "table_name": name,
        "row_count": len(df),
        "column_count": df.shape[1],
    }
    if key_col and key_col in df.columns:
        row["key_col"] = key_col
        row["unique_key_count"] = df[key_col].nunique(dropna=True)
        row["duplicate_key_rows"] = int(df.duplicated(subset=[key_col]).sum())
    return row


raw_overview = pd.DataFrame(
    [
        summarize_table("orders", orders, "order_id"),
        summarize_table("order_items", order_items),
        summarize_table("payments", payments),
        summarize_table("reviews", reviews),
        summarize_table("customers", customers, "customer_id"),
        summarize_table("sellers", sellers, "seller_id"),
        summarize_table("products", products, "product_id"),
        summarize_table("translation", translation, "product_category_name"),
        summarize_table("geolocation", geolocation),
    ]
)

print("Tổng quan dữ liệu thô:")
display(raw_overview)

print("\nPhân bố trạng thái đơn hàng:")
display(orders["order_status"].value_counts(dropna=False).rename_axis("order_status").reset_index(name="count"))

print("\nẢnh chụp nhanh giá trị thiếu:")
display(
    pd.DataFrame(
        {
            "orders_missing": orders.isna().sum(),
            "products_missing": products.reindex(columns=products.columns).isna().sum(),
        }
    ).fillna(0)
)


Tổng quan dữ liệu thô:


,table_name,row_count,column_count,key_col,unique_key_count,duplicate_key_rows
0,orders,99441,8,order_id,99441.0,0.0
1,order_items,112650,7,NaN,NaN,NaN
2,payments,103886,5,NaN,NaN,NaN
3,reviews,99224,7,NaN,NaN,NaN
4,customers,99441,5,customer_id,99441.0,0.0
5,sellers,3095,4,seller_id,3095.0,0.0
6,products,32951,9,product_id,32951.0,0.0
7,translation,71,2,product_category_name,71.0,0.0
8,geolocation,1000163,5,NaN,NaN,NaN



Phân bố trạng thái đơn hàng:


,order_status,count
0,delivered,96478
1,shipped,1107
2,canceled,625
3,unavailable,609
4,invoiced,314
5,processing,301
6,created,5
7,approved,2



Ảnh chụp nhanh giá trị thiếu:


,orders_missing,products_missing
customer_id,0.0,0.0
order_approved_at,160.0,0.0
order_delivered_carrier_date,1783.0,0.0
order_delivered_customer_date,2965.0,0.0
order_estimated_delivery_date,0.0,0.0
order_id,0.0,0.0
order_purchase_timestamp,0.0,0.0
order_status,0.0,0.0
product_category_name,0.0,610.0
product_description_lenght,0.0,610.0


## 4. Tạo dimension cho geolocation, customers, sellers và products

Ý tưởng:
- Nén geolocation về cấp `zip_code_prefix`
- Join tọa độ vào customers và sellers
- Đổi category sang tiếng Anh cho products


In [5]:
# 4.1. Nén bảng geolocation về cấp zip.
# Dùng median cho lat/lng để giảm ảnh hưởng của outlier.
dim_geo_zip = (
    geolocation.groupby("geolocation_zip_code_prefix", as_index=False)
    .agg(
        geo_lat=("geolocation_lat", "median"),
        geo_lng=("geolocation_lng", "median"),
        geo_city=("geolocation_city", safe_mode),
        geo_state=("geolocation_state", safe_mode),
        geo_point_count=("geolocation_lat", "size"),
    )
    .rename(columns={"geolocation_zip_code_prefix": "zip_code_prefix"})
)

# 4.2. Gắn tọa độ vào customers.
dim_customers_geo = customers.merge(
    dim_geo_zip,
    left_on="customer_zip_code_prefix",
    right_on="zip_code_prefix",
    how="left",
)
dim_customers_geo = dim_customers_geo.rename(
    columns={
        "geo_lat": "customer_lat",
        "geo_lng": "customer_lng",
        "geo_city": "customer_geo_city",
        "geo_state": "customer_geo_state",
        "geo_point_count": "customer_geo_point_count",
    }
).drop(columns=["zip_code_prefix"])

# 4.3. Gắn tọa độ vào sellers.
dim_sellers_geo = sellers.merge(
    dim_geo_zip,
    left_on="seller_zip_code_prefix",
    right_on="zip_code_prefix",
    how="left",
)
dim_sellers_geo = dim_sellers_geo.rename(
    columns={
        "geo_lat": "seller_lat",
        "geo_lng": "seller_lng",
        "geo_city": "seller_geo_city",
        "geo_state": "seller_geo_state",
        "geo_point_count": "seller_geo_point_count",
    }
).drop(columns=["zip_code_prefix"])

# 4.4. Chuẩn hóa category cho products.
# Nếu không có translation, giữ tên category gốc để tránh mất dữ liệu.
dim_products = products.merge(
    translation,
    on="product_category_name",
    how="left",
)
dim_products["product_category_final"] = dim_products["product_category_name_english"].fillna(
    dim_products["product_category_name"]
)

print("dim_geo_zip:", dim_geo_zip.shape)
print("dim_customers_geo:", dim_customers_geo.shape)
print("dim_sellers_geo:", dim_sellers_geo.shape)
print("dim_products:", dim_products.shape)


dim_geo_zip: (19015, 6)
dim_customers_geo: (99441, 10)
dim_sellers_geo: (3095, 9)
dim_products: (32951, 11)


## 5. Aggregate payments và reviews về cấp order

Đây là bước bắt buộc để tránh đếm trùng khi về sau join với `order_items`.


In [6]:
# 5.1. Aggregate payments.
# payment_type_main được chọn theo tổng payment_value lớn nhất trong mỗi order.
payment_type_rank = (
    payments.groupby(["order_id", "payment_type"], as_index=False)["payment_value"]
    .sum()
    .sort_values(["order_id", "payment_value", "payment_type"], ascending=[True, False, True])
)
payment_type_main = payment_type_rank.drop_duplicates(subset=["order_id"])[["order_id", "payment_type"]]
payment_type_main = payment_type_main.rename(columns={"payment_type": "payment_type_main"})

agg_payments = payments.groupby("order_id", as_index=False).agg(
    payment_value_total=("payment_value", "sum"),
    payment_record_count=("payment_sequential", "count"),
    payment_sequential_max=("payment_sequential", "max"),
    payment_installments_max=("payment_installments", "max"),
    payment_installments_mean=("payment_installments", "mean"),
)
agg_payments = agg_payments.merge(payment_type_main, on="order_id", how="left")

# 5.2. Aggregate reviews.
# Trước tiên sắp xếp để lấy review mới nhất của mỗi order.
reviews_sorted = reviews.sort_values(
    by=["order_id", "review_answer_timestamp", "review_creation_date"],
    ascending=[True, True, True],
    na_position="last",
)
latest_reviews = reviews_sorted.groupby("order_id", as_index=False).tail(1)[
    ["order_id", "review_score", "review_comment_title", "review_comment_message"]
]
latest_reviews = latest_reviews.rename(
    columns={
        "review_score": "latest_review_score",
        "review_comment_title": "latest_review_title",
        "review_comment_message": "latest_review_message",
    }
)

agg_reviews = reviews.groupby("order_id", as_index=False).agg(
    review_count=("review_id", "count"),
    review_score_mean=("review_score", "mean"),
    review_score_min=("review_score", "min"),
    review_score_max=("review_score", "max"),
    review_has_title=("review_comment_title", lambda s: s.notna().any()),
    review_has_message=("review_comment_message", lambda s: s.notna().any()),
    review_message_count=("review_comment_message", lambda s: s.notna().sum()),
    review_comment_length_mean=(
        "review_comment_message",
        lambda s: s.dropna().astype(str).str.len().mean(),
    ),
    review_creation_date_min=("review_creation_date", "min"),
    review_answer_timestamp_max=("review_answer_timestamp", "max"),
)
agg_reviews = agg_reviews.merge(latest_reviews, on="order_id", how="left")
agg_reviews["review_response_days"] = (
    agg_reviews["review_answer_timestamp_max"] - agg_reviews["review_creation_date_min"]
).dt.total_seconds() / 86400

print("agg_payments:", agg_payments.shape)
print("agg_reviews:", agg_reviews.shape)


agg_payments: (99440, 7)
agg_reviews: (98673, 15)


## 6. Tạo mart_order_level

Bảng này là nền cho logistics, review, và KPI cấp đơn hàng.
Mỗi dòng phải tương ứng với 1 `order_id`.


In [7]:
# Merge theo trục trung tâm là orders.
mart_order_level = orders.merge(
    dim_customers_geo,
    on="customer_id",
    how="left",
).merge(
    agg_payments,
    on="order_id",
    how="left",
).merge(
    agg_reviews,
    on="order_id",
    how="left",
)

# Sinh các cột thời gian và logistics để dùng lại ở giai đoạn 2 và Tableau.
mart_order_level["purchase_date"] = mart_order_level["order_purchase_timestamp"].dt.date
mart_order_level["purchase_year"] = mart_order_level["order_purchase_timestamp"].dt.year
mart_order_level["purchase_month"] = mart_order_level["order_purchase_timestamp"].dt.month
mart_order_level["purchase_year_month"] = mart_order_level["order_purchase_timestamp"].dt.to_period("M").astype(str)
mart_order_level["purchase_weekday"] = mart_order_level["order_purchase_timestamp"].dt.day_name()
mart_order_level["is_weekend_order"] = mart_order_level["order_purchase_timestamp"].dt.weekday >= 5
mart_order_level["is_delivered"] = mart_order_level["order_status"].eq("delivered")

mart_order_level["approval_lag_days"] = (
    mart_order_level["order_approved_at"] - mart_order_level["order_purchase_timestamp"]
).dt.total_seconds() / 86400
mart_order_level["carrier_lag_days"] = (
    mart_order_level["order_delivered_carrier_date"] - mart_order_level["order_approved_at"]
).dt.total_seconds() / 86400
mart_order_level["shipping_days"] = (
    mart_order_level["order_delivered_customer_date"] - mart_order_level["order_purchase_timestamp"]
).dt.total_seconds() / 86400
mart_order_level["delivery_delay_days"] = (
    mart_order_level["order_delivered_customer_date"] - mart_order_level["order_estimated_delivery_date"]
).dt.total_seconds() / 86400
mart_order_level["is_delayed"] = mart_order_level["delivery_delay_days"] > 0

# Có thể thêm flag đánh giá tích cực để dùng cho storytelling sau này.
mart_order_level["is_positive_review"] = mart_order_level["review_score_mean"] >= 4

print("mart_order_level shape:", mart_order_level.shape)
print("unique order_id:", mart_order_level["order_id"].nunique())


mart_order_level shape: (99441, 50)
unique order_id: 99441


## 7. Tạo mart_item_level

Bảng này là nền cho sales analysis, category analysis, và seller analysis.
Mỗi dòng phải tương ứng với 1 item trong đơn hàng.


In [8]:
def haversine_km(lat1, lng1, lat2, lng2):
    """Tính khoảng cách đường chim bay giữa hai tọa độ, đơn vị km."""
    lat1 = np.radians(lat1)
    lng1 = np.radians(lng1)
    lat2 = np.radians(lat2)
    lng2 = np.radians(lng2)

    dlat = lat2 - lat1
    dlng = lng2 - lng1
    a = np.sin(dlat / 2.0) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlng / 2.0) ** 2
    c = 2 * np.arcsin(np.sqrt(a))
    earth_radius_km = 6371.0
    return earth_radius_km * c


# Chọn một tập cột order-level cần thiết để đưa vào item mart.
# Lưu ý: có đưa payment/review vào đây để tham chiếu, nhưng không nên aggregate KPI cấp order trên item mart nếu không deduplicate.
order_level_columns = [
    "order_id",
    "customer_id",
    "order_status",
    "order_purchase_timestamp",
    "purchase_date",
    "purchase_year",
    "purchase_month",
    "purchase_year_month",
    "purchase_weekday",
    "is_weekend_order",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
    "shipping_days",
    "delivery_delay_days",
    "is_delayed",
    "payment_value_total",
    "payment_type_main",
    "review_score_mean",
    "latest_review_score",
    "customer_state",
    "customer_city",
    "customer_zip_code_prefix",
    "customer_lat",
    "customer_lng",
]

mart_item_level = order_items.merge(
    mart_order_level[order_level_columns],
    on="order_id",
    how="left",
).merge(
    dim_products,
    on="product_id",
    how="left",
).merge(
    dim_sellers_geo,
    on="seller_id",
    how="left",
)

# Sinh metric ở cấp item.
mart_item_level["item_revenue"] = mart_item_level["price"]
mart_item_level["item_total"] = mart_item_level["price"] + mart_item_level["freight_value"]
mart_item_level["freight_ratio"] = np.where(
    mart_item_level["price"].gt(0),
    mart_item_level["freight_value"] / mart_item_level["price"],
    np.nan,
)

# Tính khoảng cách seller-customer nếu đủ tọa độ.
has_coordinates = (
    mart_item_level[["customer_lat", "customer_lng", "seller_lat", "seller_lng"]]
    .notna()
    .all(axis=1)
)
mart_item_level["seller_customer_distance_km"] = np.nan
mart_item_level.loc[has_coordinates, "seller_customer_distance_km"] = haversine_km(
    mart_item_level.loc[has_coordinates, "customer_lat"],
    mart_item_level.loc[has_coordinates, "customer_lng"],
    mart_item_level.loc[has_coordinates, "seller_lat"],
    mart_item_level.loc[has_coordinates, "seller_lng"],
)

print("mart_item_level shape:", mart_item_level.shape)
print("unique order_id in item mart:", mart_item_level["order_id"].nunique())


mart_item_level shape: (112650, 52)
unique order_id in item mart: 98666


## 8. Validation và export

Cell cuối cùng sẽ:
- tạo bảng validation tổng hợp
- xuất tất cả bảng ra `clean_data/`
- giúp bạn đối chiếu nhanh sau khi chạy


In [9]:
def build_validation_row(name: str, df: pd.DataFrame, key_col: str | None = None, important_cols: list[str] | None = None) -> dict:
    row = {
        "table_name": name,
        "row_count": len(df),
        "column_count": df.shape[1],
    }
    if key_col and key_col in df.columns:
        row["key_col"] = key_col
        row["unique_key_count"] = df[key_col].nunique(dropna=True)
        row["duplicate_key_rows"] = int(df.duplicated(subset=[key_col]).sum())
    if important_cols:
        for col in important_cols:
            if col in df.columns:
                row[f"null_pct__{col}"] = round(df[col].isna().mean() * 100, 2)
    return row


validation_summary = pd.DataFrame(
    [
        build_validation_row("dim_geo_zip", dim_geo_zip, "zip_code_prefix", ["geo_lat", "geo_lng"]),
        build_validation_row("dim_customers_geo", dim_customers_geo, "customer_id", ["customer_lat", "customer_lng"]),
        build_validation_row("dim_sellers_geo", dim_sellers_geo, "seller_id", ["seller_lat", "seller_lng"]),
        build_validation_row("dim_products", dim_products, "product_id", ["product_category_final"]),
        build_validation_row("agg_payments", agg_payments, "order_id", ["payment_value_total", "payment_type_main"]),
        build_validation_row("agg_reviews", agg_reviews, "order_id", ["review_score_mean", "latest_review_score"]),
        build_validation_row("mart_order_level", mart_order_level, "order_id", ["customer_lat", "payment_value_total", "review_score_mean"]),
        build_validation_row("mart_item_level", mart_item_level, None, ["product_category_final", "seller_lat", "seller_customer_distance_km"]),
    ]
)

display(validation_summary)

# Tạo dictionary để export tất cả output một cách ngắn gọn.
tables_to_export = {
    "dim_geo_zip.csv": dim_geo_zip,
    "dim_customers_geo.csv": dim_customers_geo,
    "dim_sellers_geo.csv": dim_sellers_geo,
    "dim_products.csv": dim_products,
    "agg_payments.csv": agg_payments,
    "agg_reviews.csv": agg_reviews,
    "mart_order_level.csv": mart_order_level,
    "mart_item_level.csv": mart_item_level,
    "phase1_validation_summary.csv": validation_summary,
}

for file_name, df in tables_to_export.items():
    output_path = OUTPUT_DIR / file_name
    df.to_csv(output_path, index=False)
    print(f"Đã lưu: {output_path}")

print("\nPhase 1 scaffold đã hoàn tất. Bạn có thể chạy lần lượt từng cell để sinh dữ liệu sạch.")


,table_name,row_count,column_count,key_col,unique_key_count,duplicate_key_rows,null_pct__geo_lat,null_pct__geo_lng,null_pct__customer_lat,null_pct__customer_lng,null_pct__seller_lat,null_pct__seller_lng,null_pct__product_category_final,null_pct__payment_value_total,null_pct__payment_type_main,null_pct__review_score_mean,null_pct__latest_review_score,null_pct__seller_customer_distance_km
0,dim_geo_zip,19015,6,zip_code_prefix,19015.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,dim_customers_geo,99441,10,customer_id,99441.0,0.0,NaN,NaN,0.28,0.28,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,dim_sellers_geo,3095,9,seller_id,3095.0,0.0,NaN,NaN,NaN,NaN,0.23,0.23,NaN,NaN,NaN,NaN,NaN,NaN
3,dim_products,32951,11,product_id,32951.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,1.85,NaN,NaN,NaN,NaN,NaN
4,agg_payments,99440,7,order_id,99440.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,NaN,NaN,NaN
5,agg_reviews,98673,15,order_id,98673.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.00,0.0,NaN
6,mart_order_level,99441,50,order_id,99441.0,0.0,NaN,NaN,0.28,NaN,NaN,NaN,NaN,0.0,NaN,0.77,NaN,NaN
7,mart_item_level,112650,52,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.22,NaN,1.42,NaN,NaN,NaN,NaN,0.49


Đã lưu: /Users/tranquangtrong/Desktop/Olist-InsightHub-From-Data-Exploration-to-Strategic-Storytelling/clean_data/dim_geo_zip.csv
Đã lưu: /Users/tranquangtrong/Desktop/Olist-InsightHub-From-Data-Exploration-to-Strategic-Storytelling/clean_data/dim_customers_geo.csv
Đã lưu: /Users/tranquangtrong/Desktop/Olist-InsightHub-From-Data-Exploration-to-Strategic-Storytelling/clean_data/dim_sellers_geo.csv
Đã lưu: /Users/tranquangtrong/Desktop/Olist-InsightHub-From-Data-Exploration-to-Strategic-Storytelling/clean_data/dim_products.csv
Đã lưu: /Users/tranquangtrong/Desktop/Olist-InsightHub-From-Data-Exploration-to-Strategic-Storytelling/clean_data/agg_payments.csv
Đã lưu: /Users/tranquangtrong/Desktop/Olist-InsightHub-From-Data-Exploration-to-Strategic-Storytelling/clean_data/agg_reviews.csv
Đã lưu: /Users/tranquangtrong/Desktop/Olist-InsightHub-From-Data-Exploration-to-Strategic-Storytelling/clean_data/mart_order_level.csv
Đã lưu: /Users/tranquangtrong/Desktop/Olist-InsightHub-From-Data-Explorat